# TenCirChem: H2O top-important doubles (frozen core)

This notebook runs **H₂O** (STO-3G, frozen O 1s) so you can play with how many UCCSD
**double excitations** are enough for chemical accuracy.

Qubit count:
- Full STO-3G H₂O: **7 spatial orbitals → 14 qubits** (10 electrons).
- Freeze O 1s core (2e / 1 orbital): `active_space=(8, 6)` → **12 qubits**.
- After the usual 2-qubit Z₂ taper (α/β spin-block parity, as in `state_transfer/taper_lib.py`):
  **12 → 10 tapered qubits**.

Tasks (same pattern as `H4.ipynb` / `HF.ipynb`):

1. At each O–H bond length, rank doubles by `|MP2 initial θ|` at that geometry
   and keep top-`k` (`top_k` below) — not one fixed ansatz for the whole scan.
2. Scan the symmetric O–H stretch (fixed ∠HOH) and plot the total energy
   curve (and error vs FCI in the active space) vs bond length.
3. Sweep `k = 1 .. n_doubles` at one geometry to see how many doubles are enough.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize_scalar

from tencirchem import UCCSD, M
from tencirchem.static.ci_utils import get_ci_strings, get_init_civector

plt.style.use("seaborn-v0_8-whitegrid")
CHEM_ACC_MHA = 1.6  # ~1 kcal/mol


In [ ]:
# H2O: symmetric stretch of both O-H bonds at fixed H-O-H angle (xz plane).
# Equilibrium ~0.96 A, 104.5 deg. Oxygen at the origin.
HOH_ANGLE_DEG = 104.5


def make_h2o(d, angle_deg=HOH_ANGLE_DEG):
    th = np.deg2rad(float(angle_deg)) / 2.0
    d = float(d)
    return M(
        atom=[
            ["O", 0.0, 0.0, 0.0],
            ["H", d * np.sin(th), 0.0, d * np.cos(th)],
            ["H", -d * np.sin(th), 0.0, d * np.cos(th)],
        ],
        basis="sto-3g",
        unit="Angstrom",
    )


# Frozen O 1s: 8e in 6 spatial orbitals -> 12 qubits. Taper later -> 10q.
# Full STO-3G would be (10e, 7 orb) -> 14 qubits.
active_space = (8, 6)
N_SPATIAL_ORBITALS = active_space[1]
N_QUBITS = 2 * N_SPATIAL_ORBITALS          # 12 (pre-taper)
N_QUBITS_TAPERED = N_QUBITS - 2            # 10
N_ACTIVE_ELECTRONS = active_space[0]
ETA = N_ACTIVE_ELECTRONS // 2              # 4 occupied active spatial / spin

# O-H bond-length scan (A); include near-equilibrium 0.96 A
d_grid = np.array([0.8, 0.96, 1.1, 1.3, 1.5, 1.7])


def _paper_hf_and_excited_bitstrings(n_qubits, n_elec):
    """Paper qubit layout: alpha sector (low MO index first), then beta sector."""
    eta = n_elec // 2
    half = n_qubits // 2

    hf = []
    for _sector in range(2):
        hf.extend(["1" if local < eta else "0" for local in range(half)])

    exc = hf.copy()
    exc[eta - 1] = "0"
    exc[eta + half - 1] = "0"
    exc[eta] = "1"
    exc[eta + half] = "1"
    return "".join(hf), "".join(exc)


def _paper_to_tencirchem_bitstring(paper_bits, n_qubits, n_elec):
    """Map paper spin-orbital bit order to TenCirChem's internal qubit order."""
    eta = n_elec // 2
    half = n_qubits // 2
    tc_bits = ["0"] * n_qubits

    for sector_start in (0, half):
        for local in range(half):
            if paper_bits[sector_start + local] != "1":
                continue
            if local < eta:
                tc_local = local + (half - eta)
            else:
                tc_local = local - eta
            tc_bits[sector_start + tc_local] = "1"

    return "".join(tc_bits)


def build_multireference_init_civector(ucc, beta):
    """Paper Eq. (6): (|HF> - beta|exc>) / sqrt(1 + beta^2) in TenCirChem CI basis."""
    if beta == 0.0:
        return None

    n_qubits = ucc.n_qubits
    n_elec = ucc.n_elec
    ci_strings = get_ci_strings(n_qubits, n_elec, ucc.hcb)

    _, paper_exc = _paper_hf_and_excited_bitstrings(n_qubits, n_elec)
    exc_bits = _paper_to_tencirchem_bitstring(paper_exc, n_qubits, n_elec)
    exc_addr = int(exc_bits, 2)
    exc_idx = int(np.where(ci_strings == exc_addr)[0][0])

    norm = 1.0 / np.sqrt(1.0 + beta**2)
    ref_ci = np.zeros(len(ci_strings), dtype=float)
    ref_ci[0] = norm
    ref_ci[exc_idx] = -beta * norm
    return ref_ci


def optimize_multireference_beta(ucc, beta_bounds=(-5.0, 5.0)):
    """min_beta <Psi_ref(beta)|H|Psi_ref(beta)> at zero UCC params."""

    def ref_energy(beta):
        saved = ucc.init_state
        ucc.init_state = build_multireference_init_civector(ucc, beta)
        e = ucc.energy(np.zeros(ucc.n_params))
        ucc.init_state = saved
        return e

    res = minimize_scalar(ref_energy, bounds=beta_bounds, method="bounded")
    return float(res.x)


def configure_ucc_initial_state(
    ucc,
    use_multireference=False,
    beta=None,
    optimize_beta=True,
    beta_bounds=(-5.0, 5.0),
):
    """Configure UCC initial state before kernel(). Default = RHF HF."""
    if not use_multireference:
        ucc.init_state = None
        return 0.0

    if beta is None and optimize_beta:
        beta = optimize_multireference_beta(ucc, beta_bounds=beta_bounds)
    elif beta is None:
        beta = 0.0

    ucc.init_state = build_multireference_init_civector(ucc, beta)
    return float(beta)


print(
    f"H2O frozen-core active space: {N_ACTIVE_ELECTRONS} electrons, "
    f"{N_SPATIAL_ORBITALS} spatial orbitals -> {N_QUBITS} qubits (eta={ETA}); "
    f"after Z2 taper -> {N_QUBITS_TAPERED} qubits"
)
print(f"Scanning O-H bond lengths (A) at angle={HOH_ANGLE_DEG} deg: {d_grid.tolist()}")


In [ ]:
# Demo: HF vs paper multi-reference initial state (default is HF)
demo_d = 0.96
demo_mol = make_h2o(demo_d)
demo_ucc = UCCSD(
    demo_mol,
    active_space=active_space,
    init_method="mp2",
    pick_ex2=False,
    sort_ex2=False,
    run_fci=True,
)

configure_ucc_initial_state(demo_ucc, use_multireference=False)
e_hf = demo_ucc.energy(np.zeros(demo_ucc.n_params))

beta_demo = configure_ucc_initial_state(demo_ucc, use_multireference=True, optimize_beta=True)
e_mr = demo_ucc.energy(np.zeros(demo_ucc.n_params))

paper_hf, paper_exc = _paper_hf_and_excited_bitstrings(demo_ucc.n_qubits, demo_ucc.n_elec)
init_demo_df = pd.DataFrame(
    [
        {
            "d_angstrom": demo_d,
            "beta_optimized": beta_demo,
            "E_ref_HF_Ha": e_hf,
            "E_ref_multireference_Ha": e_mr,
            "E_FCI_Ha": demo_ucc.e_fci,
            "HF_minus_FCI_mHa": (e_hf - demo_ucc.e_fci) * 1000,
            "multiref_minus_FCI_mHa": (e_mr - demo_ucc.e_fci) * 1000,
            "paper_HF_bitstring": paper_hf,
            "paper_excited_bitstring": paper_exc,
            "tc_excited_bitstring": _paper_to_tencirchem_bitstring(
                paper_exc, demo_ucc.n_qubits, demo_ucc.n_elec
            ),
        }
    ]
)

print(f"Initial-state demo at d = {demo_d:.2f} A")
display(init_demo_df)
configure_ucc_initial_state(demo_ucc, use_multireference=False)


In [ ]:
def build_reduced_topk_doubles_ucc(mol, active_space, k, fixed_pids=None):
    """Build a UCCSD instance that keeps top-k MP2-important double parameters only.

    If ``fixed_pids`` is given, those exact double parameter ids are kept instead of
    re-ranking by MP2 amplitude at this geometry. Kept amplitudes are initialized
    from the MP2 guess at *this* molecule/geometry.
    """
    probe = UCCSD(
        mol,
        active_space=active_space,
        init_method="mp2",
        pick_ex2=False,
        sort_ex2=False,
        run_fci=False,
    )

    all_ops = probe.ex_ops
    all_param_ids = probe.param_ids
    all_init_guess = probe.init_guess

    pid_to_ops = {}
    for op, pid in zip(all_ops, all_param_ids):
        pid_to_ops.setdefault(pid, []).append(op)

    double_pids = [pid for pid, ops in pid_to_ops.items() if all(len(op) == 4 for op in ops)]
    if len(double_pids) == 0:
        raise ValueError("No double-excitation parameters found.")

    pid_to_guess = {pid: all_init_guess[pid] for pid in double_pids}
    if fixed_pids is None:
        sorted_pids = sorted(double_pids, key=lambda pid: abs(pid_to_guess[pid]), reverse=True)
        selected_pids = sorted_pids[:k]
    else:
        selected_pids = list(fixed_pids)

    selected_pids_set = set(selected_pids)
    selected_ex_ops = []
    selected_param_ids = []
    selected_pid_to_ops = {pid: [] for pid in selected_pids}
    for op, pid in zip(all_ops, all_param_ids):
        if pid in selected_pids_set:
            selected_ex_ops.append(op)
            selected_param_ids.append(pid)
            selected_pid_to_ops[pid].append(op)

    pid_remap = {old_pid: new_pid for new_pid, old_pid in enumerate(selected_pids)}
    selected_param_ids = [pid_remap[pid] for pid in selected_param_ids]
    # Use MP2 amplitudes from this geometry (not zeros / not a fixed reference d).
    selected_init_guess = [pid_to_guess[pid] for pid in selected_pids]

    reduced = UCCSD(
        mol,
        active_space=active_space,
        init_method="zeros",
        pick_ex2=False,
        sort_ex2=False,
        run_fci=False,
    )
    reduced.ex_ops = selected_ex_ops
    reduced.param_ids = selected_param_ids
    reduced.init_guess = selected_init_guess

    def op_to_string(op):
        p, q, r, s = op
        return f"a{p}dag a{q}dag a{r} a{s}"

    selected_rows = []
    for rank, pid in enumerate(selected_pids, start=1):
        linked_ops = selected_pid_to_ops[pid]
        selected_rows.append(
            {
                "rank": rank,
                "param_id_index": pid,
                "factor_type": "double",
                "mp2_init_theta": pid_to_guess[pid],
                "abs_mp2_init_theta": abs(pid_to_guess[pid]),
                "n_ex_ops_linked": len(linked_ops),
                "op_terms": " | ".join(op_to_string(op) for op in linked_ops),
            }
        )

    selected_meta_df = pd.DataFrame(selected_rows)
    return reduced, selected_pids, selected_ex_ops, selected_meta_df


# <-- play with this: number of MP2-ranked doubles kept in the bond scan
top_k = 8
scan_rows = []

# At each bond length: re-rank doubles by |MP2 θ| and seed with that geometry's MP2.
report_d = 0.96
selected_meta_df = None

for d in d_grid:
    mol = make_h2o(d)

    # Full STO-3G (no frozen core) for reference.
    ucc_no_as = UCCSD(
        mol,
        init_method="mp2",
        pick_ex2=True,
        sort_ex2=True,
        run_fci=False,
    )
    e_no_as = ucc_no_as.kernel()

    # Frozen-core active space.
    ucc_as = UCCSD(
        mol,
        active_space=active_space,
        init_method="mp2",
        pick_ex2=True,
        sort_ex2=True,
        run_fci=True,
    )
    e_as = ucc_as.kernel()
    e_fci_as = ucc_as.e_fci
    e_hf = ucc_as.e_hf

    reduced_ucc_d, selected_pids_d, selected_ex_ops_d, meta_d = build_reduced_topk_doubles_ucc(
        mol, active_space, top_k
    )
    e_topk_doubles = reduced_ucc_d.kernel()
    if abs(d - report_d) < 1e-12:
        selected_meta_df = meta_d

    scan_rows.append(
        {
            "d_angstrom": d,
            "E_HF_Ha": e_hf,
            "E_no_active_space_Ha": e_no_as,
            "E_active_space_Ha": e_as,
            "E_topk_doubles_Ha": e_topk_doubles,
            "topk_minus_active_mHa": (e_topk_doubles - e_as) * 1000,
            "topk_minus_fci_active_mHa": (e_topk_doubles - e_fci_as) * 1000,
            "n_ex_ops_kept": len(selected_ex_ops_d),
            "selected_param_ids": str(selected_pids_d),
            "mp2_init_thetas": str([round(float(x), 6) for x in meta_d["mp2_init_theta"].tolist()]),
        }
    )

scan_df = pd.DataFrame(scan_rows).sort_values("d_angstrom").reset_index(drop=True)

summary_df = scan_df[
    [
        "d_angstrom",
        "E_HF_Ha",
        "E_no_active_space_Ha",
        "E_active_space_Ha",
        "E_topk_doubles_Ha",
        "topk_minus_active_mHa",
        "topk_minus_fci_active_mHa",
        "selected_param_ids",
        "mp2_init_thetas",
    ]
].copy()

print(
    f"Top-{top_k} doubles are re-ranked by |MP2 initial theta| at each bond length "
    f"and seeded with that geometry's MP2 amplitudes."
)
print(f"Detailed top-{top_k} doubles below are for d = {report_d:.2f} A")
print(f"Pre-taper qubits = {N_QUBITS}; after Z2 taper = {N_QUBITS_TAPERED}")
display(selected_meta_df)
display(summary_df)


In [ ]:
# Convergence vs number of doubles kept — the main "how many is enough?" cell.
# Change conv_d or re-run after editing; sweeps k = 1 .. n_doubles_total.
# (8,6) has 42 doubles; this can take a few minutes.

conv_d = 0.96
conv_mol = make_h2o(conv_d)

conv_ref = UCCSD(
    conv_mol,
    active_space=active_space,
    init_method="mp2",
    pick_ex2=True,
    sort_ex2=True,
    run_fci=True,
)
e_full_conv = conv_ref.kernel()
e_fci_conv = conv_ref.e_fci

probe_conv = UCCSD(
    conv_mol,
    active_space=active_space,
    init_method="mp2",
    pick_ex2=False,
    sort_ex2=False,
    run_fci=False,
)
_pid_ops = {}
for op, pid in zip(probe_conv.ex_ops, probe_conv.param_ids):
    _pid_ops.setdefault(pid, []).append(op)
n_doubles_total = sum(1 for ops in _pid_ops.values() if all(len(o) == 4 for o in ops))

conv_rows = []
for k in range(1, n_doubles_total + 1):
    red_k, _, ex_ops_k, _ = build_reduced_topk_doubles_ucc(conv_mol, active_space, k)
    e_k = red_k.kernel()
    conv_rows.append(
        {
            "n_doubles_kept": k,
            "n_ex_ops": len(ex_ops_k),
            "E_reduced_Ha": e_k,
            "error_vs_FCI_mHa": (e_k - e_fci_conv) * 1000,
        }
    )
    if k % 5 == 0 or k == n_doubles_total:
        print(f"  k={k}/{n_doubles_total}: error = {(e_k - e_fci_conv) * 1000:.3f} mHa")

conv_df = pd.DataFrame(conv_rows)
enough = conv_df.loc[conv_df["error_vs_FCI_mHa"] <= CHEM_ACC_MHA, "n_doubles_kept"]
k_chem = int(enough.iloc[0]) if len(enough) else None

print(
    f"Convergence demo at d = {conv_d:.2f} A  "
    f"(no={N_ACTIVE_ELECTRONS // 2}, nv={N_SPATIAL_ORBITALS - N_ACTIVE_ELECTRONS // 2}, "
    f"total doubles={n_doubles_total})"
)
print(f"HF  = {conv_ref.e_hf:.8f} Ha")
print(f"FCI = {e_fci_conv:.8f} Ha   full UCCSD = {e_full_conv:.8f} Ha")
if k_chem is not None:
    print(f"First k with error <= {CHEM_ACC_MHA} mHa: k = {k_chem}")
else:
    print(f"No k reached chemical accuracy ({CHEM_ACC_MHA} mHa) in this sweep.")
display(conv_df)

ax = conv_df.plot(
    x="n_doubles_kept",
    y="error_vs_FCI_mHa",
    marker="o",
    legend=False,
    figsize=(7, 4),
)
ax.axhline(0.0, color="black", linestyle="--", linewidth=1.0)
ax.axhline(CHEM_ACC_MHA, color="tab:green", linestyle="--", linewidth=1.0, label=f"{CHEM_ACC_MHA} mHa")
ax.axvline(top_k, color="tab:red", linestyle=":", linewidth=1.0, label=f"top-{top_k}")
ax.set_xlabel("number of important doubles kept (k)")
ax.set_ylabel("error vs FCI (mHa)")
ax.set_title(f"H2O (8,6) / 12q (→10 tapered): convergence at d={conv_d:.2f} A")
ax.legend()
plt.show()


In [ ]:
display(scan_df)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Total energy curve vs O-H bond length
scan_df.plot(
    x="d_angstrom",
    y=["E_HF_Ha", "E_no_active_space_Ha", "E_active_space_Ha", "E_topk_doubles_Ha"],
    marker="o",
    ax=axes[0],
)
axes[0].set_xlabel("O-H bond length (A)")
axes[0].set_ylabel("total energy (Ha)")
axes[0].set_title("H2O total energy vs O-H stretch")
axes[0].legend(["HF", "full STO-3G UCCSD", "frozen-core UCCSD", f"top-{top_k} doubles"])

# Error vs FCI in active space
scan_df.plot(
    x="d_angstrom",
    y="topk_minus_fci_active_mHa",
    marker="o",
    legend=False,
    ax=axes[1],
)
axes[1].axhline(CHEM_ACC_MHA, color="black", linestyle="--", linewidth=1.0)
axes[1].set_xlabel("O-H bond length (A)")
axes[1].set_ylabel(f"E(top-{top_k} doubles) - E(FCI active) (mHa)")
axes[1].set_title(f"H2O top-{top_k} doubles accuracy (12q → 10 tapered)")

plt.tight_layout()
plt.show()


## Notes

- Molecule is `H2O` in STO-3G with symmetric O–H stretch at fixed ∠HOH = 104.5°
  via `make_h2o(d)`.
- Freeze O 1s: `active_space=(8, 6)` → **12 qubits** (8 active electrons in 6 spatial orbitals).
  Full STO-3G without freeze is 10e / 7 orb → 14 qubits (`E_no_active_space_Ha`).
- Z₂ qubit taper (α/β spin-block parity, same as `state_transfer/taper_lib.py`):
  **12 → 10 qubits**. This notebook selects doubles on the **12-qubit** register.
- Top-`k` selection (`top_k`) is **doubles only**, re-ranked by `|MP2 initial θ|`
  **at each bond length** (not a single fixed ansatz across the scan).
- Active space `(8, 6)` has `no = 4`, `nv = 2` → **42 double parameters** (plus 8 singles).
  The convergence cell sweeps every `k` so you can see how many doubles are enough
  for ≤ 1.6 mHa vs FCI in the frozen-core active space.
